In [1]:
import torch
import torch.nn as nn 
import torchvision 
import torch.optim as optim
from torchvision.datasets import CIFAR10


In [2]:
# Datasets and DataLoader 
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root = "./data",train =True ,download =True,transform=transform)
testset = CIFAR10(root ="./data",train=False , download = True , transform = transform)
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [3]:
train_loader = DataLoader(trainset , batch_size  = 64 ,shuffle =True)
test_loader = DataLoader(testset,batch_size = 64 )

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32,kernel_size = 3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel size  =2,stride = 2

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size = 3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
    )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1)
        x = self.fc_layers(x)
        return x
        

In [5]:
model = CNN()

In [6]:
crietrion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [7]:
# Training 

epochs = 12
model.train()
for epoch in range(epochs):
    training_loss = 0
    for imgs , labels in train_loader:
        optimizer.zero_grad()   
        output = model.forward(imgs)
        loss = crietrion(output,labels)  #loss fxn
        loss.backward()  #BP
        optimizer.step() #update params

        training_loss += loss.item()
    print(f"totall training loss for epoch{epoch+1}/{epochs} : {training_loss/len(train_loader)}")


totall training loss for epoch1/12 : 1.3776899155448465
totall training loss for epoch2/12 : 0.9404244984659698
totall training loss for epoch3/12 : 0.7589821376459068
totall training loss for epoch4/12 : 0.6382090230960675
totall training loss for epoch5/12 : 0.5351595193757426
totall training loss for epoch6/12 : 0.4436883049471604
totall training loss for epoch7/12 : 0.36048131201730665
totall training loss for epoch8/12 : 0.28299246152953417
totall training loss for epoch9/12 : 0.21746820028480665
totall training loss for epoch10/12 : 0.17603657180038484
totall training loss for epoch11/12 : 0.13777317710535225
totall training loss for epoch12/12 : 0.12047987104014819


In [9]:
# testing
correct_pred = 0
total_labels = 0
model.eval()
with torch.no_grad():
    for imgs , labels in test_loader:
        outputs = model.forward(imgs)
        _,predicted = torch.max(outputs,1)
        correct_pred +=(predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy of model is  : {(correct_pred/total_labels)*100}")
    

Accuracy of model is  : 74.95
